In [89]:
import openrouteservice as ors
import pandas as pd
client = ors.Client(key='5b3ce3597851110001cf6248f1a1d4347a304d829546c3d3c0e56ced')

Read in curated data

In [90]:
rent_data = pd.read_csv("../data/raw/rental_data.csv")
metro_stations_data = pd.read_csv("../data/raw/PTV/stations_with_postcodes.csv")
regional_stations_data = pd.read_csv("../data/raw/PTV/stations_with_postcodes.csv")

stations_data = metro_stations_data.append(regional_stations_data, ignore_index=True)

/var/folders/l_/flqtngrd05d8qnbgmbtshrx00000gn/T/ipykernel_25704/2786767506.py:6: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  stations_data = metro_stations_data.append(regional_stations_data, ignore_index=True)


Calculate the haversine distance between the coordinates of houses and train stations

In [95]:
import pandas as pd
import math

# 'coordinates' column in rent_data contains lists of [latitude, longitude]
rent_data['latitude'] = rent_data['coordinates'].apply(lambda x: float(x.split(',')[0].strip('[ ]')))
rent_data['longitude'] = rent_data['coordinates'].apply(lambda x: float(x.split(',')[1].strip('[ ]')))

# Function to calculate Haversine distance between two sets of coordinates
def haversine_distance(lat1, lon1, lat2, lon2):
    """ Calculates the haversine distance between 2 coordinates in terms of lattitude and longitude and returns the distance in kms"""
    # Radius of the Earth in kilometers
    earth_radius = 6371.0

    # Convert degrees to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    # Haversine formula
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = earth_radius * c

    return distance

# Create an empty list to store the minimum distances
min_distances = []

# Iterate over each house in rent_data
for house_idx, house_row in rent_data.iterrows():
    house_lat = house_row['latitude']
    house_lon = house_row['longitude']
    nearest_distance = float('inf')

    # Iterate over each train station in station data
    for station_idx, station_row in stations_data.iterrows():
        station_lat = station_row['LATITUDE']
        station_lon = station_row['LONGITUDE']

        # Calculate Haversine distance
        distance = haversine_distance(house_lat, house_lon, station_lat, station_lon)

        # If this distance is smaller than the previously recorded nearest_distance,
        if distance < nearest_distance:
            nearest_distance = distance

    min_distances.append(nearest_distance)

# Add the minimum distances and corresponding postcodes to the rent_data DataFrame
rent_data['nearest train station'] = min_distances

# Save the updated DataFrame to a CSV file
rent_data.to_csv("../data/curated/rental_data_train_station.csv", index=False)


In [93]:
rent_data

,name,cost_text,coordinates,property type,beds,baths,parking,title,description,property features,NBN,Weekly Rent Amount,Postcode,latitude,longitude,Nearest Distance (km)
0,Rooms/227 Elgar Road Surrey Hills VIC 3127,$120.00,"[-37.8353315, 145.1114189]",House,1,2,6,Single room in rooming house - CURRENTLY NO RO...,"['** BOOK INSPECTION ONLINE **', '*Currently n...","['Furnished*', 'Broadband internet access*', '...",No NBN,120,3127,-37.835332,145.111419,1.667194
1,30 Daly Boulevard Highton VIC 3216,$130,"[-38.1962932, 144.3060777]",House,9,2,2,Student Accommodation - 1 Room Available!,['We have 1 Rooms Available to Rent.Room 2 - $...,"['Gas*', 'Internal Laundry*', 'Secure Parking*...",nbn® Fibre to the Node (FTTN) is available in ...,130,3216,-38.196293,144.306078,45.349305
2,Berth 5274 Martha Cove Waterway Safety Beach V...,$140 per week,"[-38.3138012, 145.0045603]",House,1,1,0,Boat Berth B5274,['BOAT BERTH AVAILABLE-B5274-Located in Martha...,[],No NBN,140,3936,-38.313801,145.004560,15.357555
3,18 Cedarville Close Highton VIC 3216,$140,"[-38.19508, 144.3154857]",House,1,2,6,Room 4 rent!,['Rooms for rent in this student share house. ...,"['Built in wardrobes', 'Internal Laundry', 'Ga...",nbn® Fibre to the Premises (FTTP) is available...,140,3216,-38.195080,144.315486,44.689377
4,123 Thornhill Road Highton VIC 3216,$150,"[-38.1845196, 144.3107906]",House,1,4,3,Room for rent near Uni,['Rooms within walking distance to Waurn Ponds...,"['Ensuite', 'Furnished', 'Built in wardrobes*'...",nbn® Fibre to the Node (FTTN) is available in ...,150,3216,-38.184520,144.310791,44.119043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11400,Enquire For Address Brighton VIC 3186,"$4,400.00","[-37.9044002, 144.9997396]",House,4,4,2,Golden Mile Luxury,"['Inspect by Appointment Only ', 'This family ...","['Study', 'Swimming Pool', 'Ducted Cooling', '...",No NBN,4400,3186,-37.904400,144.999740,0.257162
11401,335 The Esplanade Indented Head VIC 3223,"$4,700 pw /$700 pn High, $3,775 pw/$545 pn Low","[-38.1439923, 144.7147899]",House,4,3,0,"335 The Esplanade, Indented Head",['This stylish 4 bedroom home will surely make...,"['Ensuite', 'Deck', 'Dishwasher', 'Ducted Heat...",No NBN,4700,3223,-38.143992,144.714790,27.603203
11402,8701/7 Riverside Quay Southbank VIC 3006,"$5,000.00","[-37.8213342, 144.9646982]",Apartment / Unit / Flat,3,3,4,On Top of the World,"[""A brilliant opportunity to lease the exclusi...",[],nbn® Fibre to the Building (FTTB) is available...,5000,3006,-37.821334,144.964698,0.391236
11403,1 Weston Street Balwyn VIC 3103,"$9,950 weekly","[-37.8117577, 145.0775507]",House,6,7,8,"Lifestyle, Modern & Luxury Living Awaits",['**Scheduling your property inspection is now...,"['Air conditioning*', 'Built in wardrobes*', '...",nbn® Hybrid Fibre Coaxial (HFC) is available i...,9950,3103,-37.811758,145.077551,1.451528


open route service - driving distance

In [94]:
# min_distances = []

# for house_idx, house_row in rent_data.iterrows():
#     house_lat = house_row['coordinates'][0]
#     house_lon = house_row['coordinates'][1]
    
#     nearest_distance = None  # Default value for cases where routing fails

#     for station_idx, station_row in stations_data.iterrows():
#         station_coord = (station_row['LATITUDE'], station_row['LONGITUDE'])

#         radius = 1000

#         # Calculate the driving distance using OpenRouteService
#         route = client.directions(
#             coordinates=[(house_lat, house_lon), station_coord],
#             profile='driving-car',
#         )

#         if 'features' in route:
#             distance = route['features'][0]['properties']['segments'][0]['distance']
#             if nearest_distance is None or distance < nearest_distance:
#                 nearest_distance = distance

#     # If nearest_distance is still None, set it to a default value (e.g., -1)
#     if nearest_distance is None:
#         nearest_distance = -1  # or any other suitable default value

#     min_distances.append(nearest_distance)